# VietEmbed-RAG V1 — VN-MTEB Retrieval Benchmark

Notebook này chỉ benchmark **Retrieval**, vì model V1 được fine-tune riêng cho RAG.

Không chạy:
- Classification
- Clustering
- STS
- Pair Classification
- Reranking

Với embedding model dùng làm retriever trong RAG, metric quan trọng nhất là chất lượng **query → relevant document retrieval**.

## Cách dùng trên Colab

1. Chọn **Runtime → Change runtime type → GPU**
2. Upload file `.zip` chứa model vào `/content/`
3. Sửa `MODEL_ZIP`
4. Chạy từ trên xuống
5. Chạy Smoke Test trước, rồi mới Full Retrieval

## 1. Cài thư viện

In [ ]:
!pip install -q -U "mteb>=2.2.0" sentence-transformers

## 2. Kiểm tra môi trường

In [ ]:
import torch
import mteb
import sentence_transformers

print("PyTorch:", torch.__version__)
print("MTEB:", mteb.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Cấu hình

Thông thường chỉ cần sửa `MODEL_ZIP`.

Nếu model V1 được train theo convention E5, giữ:

```python
USE_E5_PREFIX = True
```

Nếu lúc fine-tune bạn bỏ hoàn toàn `query:` / `passage:`, có thể benchmark thêm một lần với `False`.

In [ ]:
MODEL_ZIP = "/content/VietEmbed-RAG-V1.zip"
EXTRACT_DIR = "/content/VietEmbed-RAG-V1"

USE_E5_PREFIX = True
BATCH_SIZE = 32

RESULT_DIR = "/content/vn_mteb_retrieval_v1_results"

## 4. Giải nén model

Cell này tự tìm `modules.json`, nên không quan trọng file zip có bọc thêm một folder hay không.

In [ ]:
from pathlib import Path
import shutil

shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
shutil.unpack_archive(MODEL_ZIP, EXTRACT_DIR)

model_files = list(Path(EXTRACT_DIR).rglob("modules.json"))

if not model_files:
    raise FileNotFoundError(
        "Không tìm thấy modules.json. Hãy kiểm tra file zip có đúng là SentenceTransformer model không."
    )

MODEL_DIR = str(model_files[0].parent)

print("Model directory:", MODEL_DIR)
print("Files:")

for path in Path(MODEL_DIR).iterdir():
    print("-", path.name)

## 5. Load model

Với retrieval theo E5:

- query → `query: ...`
- document → `passage: ...`

In [ ]:
from sentence_transformers import SentenceTransformer

prompts = {}

if USE_E5_PREFIX:
    prompts = {
        "query": "query: ",
        "document": "passage: ",
    }

model = SentenceTransformer(
    MODEL_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
    prompts=prompts,
)

print(model)
print("Embedding dimension:", model.get_sentence_embedding_dimension())
print("Max sequence length:", model.max_seq_length)
print("Prompts:", prompts)

## 6. Sanity check

Kiểm tra model encode bình thường và một query gần document liên quan hơn document không liên quan.

In [ ]:
from sentence_transformers.util import cos_sim

if USE_E5_PREFIX:
    texts = [
        "query: Trí tuệ nhân tạo là gì?",
        "passage: Trí tuệ nhân tạo là lĩnh vực nghiên cứu các hệ thống có khả năng thực hiện những nhiệm vụ cần trí thông minh.",
        "passage: Hôm nay thời tiết tại thành phố có mưa lớn.",
    ]
else:
    texts = [
        "Trí tuệ nhân tạo là gì?",
        "Trí tuệ nhân tạo là lĩnh vực nghiên cứu các hệ thống có khả năng thực hiện những nhiệm vụ cần trí thông minh.",
        "Hôm nay thời tiết tại thành phố có mưa lớn.",
    ]

embeddings = model.encode(texts)

relevant_score = cos_sim(embeddings[0], embeddings[1]).item()
unrelated_score = cos_sim(embeddings[0], embeddings[2]).item()

print("Embedding shape:", embeddings.shape)
print("Relevant similarity:", relevant_score)
print("Unrelated similarity:", unrelated_score)

## 7. Load VN-MTEB Retrieval tasks

Chỉ lấy các task thuộc loại `Retrieval`.

In [ ]:
benchmark = mteb.get_benchmark("VN-MTEB (vie, v1)")

retrieval_tasks = mteb.filter_tasks(
    benchmark,
    task_types=["Retrieval"],
)

print("Number of Retrieval tasks:", len(retrieval_tasks))

for task in retrieval_tasks:
    print("-", task.metadata.name)

## 8. Result Cache

Kết quả từng task sẽ được lưu lại.

Nếu Colab bị ngắt, bạn có thể giữ thư mục kết quả hoặc zip lại trước khi thoát.
Khi chạy lại với `overwrite_strategy="only-missing"`, các phần đã có sẽ không cần chạy lại.

In [ ]:
from pathlib import Path

Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)

cache = mteb.ResultCache(
    cache_path=RESULT_DIR
)

print("Result directory:", RESULT_DIR)

# 9. Retrieval Smoke Test

Chạy trước vài corpus nhỏ / vừa để kiểm tra pipeline.

Không cần lao thẳng vào MSMARCO, HotpotQA hay các corpus hàng triệu document ngay từ đầu.

In [ ]:
SMOKE_TASKS = [
    "SciFact-VN",
    "NFCorpus-VN",
    "ArguAna-VN",
]

smoke_tasks = mteb.get_tasks(
    tasks=SMOKE_TASKS
)

mteb.evaluate(
    model,
    tasks=smoke_tasks,
    cache=cache,
    encode_kwargs={
        "batch_size": BATCH_SIZE,
    },
    overwrite_strategy="only-missing",
)

## 10. Xem kết quả Smoke Test

In [ ]:
smoke_results = cache.load_results(
    tasks=smoke_tasks,
    include_remote=False,
)

smoke_df = smoke_results.to_dataframe(
    format="long"
)

display(smoke_df)

# 11. Full VN-MTEB Retrieval

Đây là benchmark chính của V1.

Một số retrieval corpus rất lớn nên cell này có thể tốn nhiều thời gian và RAM/VRAM.

Nếu gặp CUDA OOM:

```python
BATCH_SIZE = 16
```

rồi chạy lại cell.

In [ ]:
mteb.evaluate(
    model,
    tasks=retrieval_tasks,
    cache=cache,
    encode_kwargs={
        "batch_size": BATCH_SIZE,
    },
    overwrite_strategy="only-missing",
)

## 12. Xem toàn bộ Retrieval results

In [ ]:
retrieval_results = cache.load_results(
    tasks=retrieval_tasks,
    include_remote=False,
)

retrieval_df = retrieval_results.to_dataframe(
    format="long"
)

display(retrieval_df)

## 13. Xuất CSV

CSV này dùng để lưu kết quả V1 và so với các version sau.

In [ ]:
CSV_PATH = "/content/VietEmbed-RAG-V1_VN-MTEB_Retrieval.csv"

retrieval_df.to_csv(
    CSV_PATH,
    index=False,
)

print("Saved:", CSV_PATH)

## 14. Đóng gói toàn bộ result cache

In [ ]:
import shutil

ZIP_PATH = "/content/VietEmbed-RAG-V1_VN-MTEB_Retrieval_results"

shutil.make_archive(
    ZIP_PATH,
    "zip",
    RESULT_DIR,
)

print("Saved:", ZIP_PATH + ".zip")

## 15. Tải kết quả về máy

In [ ]:
from google.colab import files

files.download("/content/VietEmbed-RAG-V1_VN-MTEB_Retrieval.csv")
files.download("/content/VietEmbed-RAG-V1_VN-MTEB_Retrieval_results.zip")

# Nên nhìn metric nào?

Với RAG retriever, ưu tiên:

**NDCG@10** → metric chính để so model.

Sau đó có thể xem thêm:

- Recall@10
- Recall@100
- MAP
- Precision

Khi làm V2, cách so sánh đơn giản nhất là:

```text
multilingual-e5-base
        ↓
VietEmbed-RAG V1
        ↓
VietEmbed-RAG V2
```

trên **cùng tập Retrieval tasks, cùng prefix convention và cùng MTEB version**.